In [2]:
from Bio import Entrez
import pandas as pd
import numpy as np
import os
import json
import time
import regex as re
import xmltodict, json
# ElementTree
import xml.etree.ElementTree as ET
from dataclasses import dataclass

from fetch_pubmed import article_machine, fetch_pubmed_data_given_ids


In [3]:
# data class for papers

@dataclass
class Paper:
    id: str
    title: str
    abstract: str
    authors: list
    journal: str
    publication_date: str
    doi: str
    keywords: list
    language_list: list
    embedding: np.array

    def __init__(self, id, title, abstract, authors, journal, publication_year, publication_month, publication_day, doi, keywords, language_list, embedding= None):
        self.id = id
        self.title = title
        self.abstract = abstract
        self.authors = authors
        self.journal = journal
        self.publication_year = publication_year
        self.publication_month = publication_month
        self.publication_day = publication_day
        self.doi = doi
        self.keywords = keywords
        self.language_list = language_list
        self.embedding = embedding

    def __str__(self):
        return f"Paper(id={self.id}, title={self.title}, abstract={self.abstract}, authors={self.authors}, journal={self.journal}, publication_year={self.publication_year}, publication_month={self.publication_month}, publication_day={self.publication_day}, doi={self.doi}, keywords={self.keywords}, language_list={self.language_list}, embedding={self.embedding})"
    
    def __repr__(self):
        return f"Paper(id={self.id}, title={self.title}, abstract={self.abstract}, authors={self.authors}, journal={self.journal}, publication_year={self.publication_year}, publication_month={self.publication_month}, publication_day={self.publication_day}, doi={self.doi}, keywords={self.keywords}, language_list={self.language_list}, embedding={self.embedding})"
    
    def save_to_json(self, filename):
        # save the paper to a json file in a nice format
        with open(filename, 'w') as f:
            json.dump(self.__dict__, f, indent=4)

    def load_from_json(self, filename):
        with open(filename, 'r') as f:
            data = json.load(f)
            return Paper(**data)
        

In [5]:
# E-Fetch runs with a maximum of ten thousand studies, we need to separate our IdList in chunks 
def fetch_pubmed_data_given_ids_in_chunks(id_list, chunk_size=10000, output_path=None):
    papers_list = []
    for i in range(0, len(id_list), chunk_size):
        print(f"Fetching papers {i} to {i+chunk_size} out of {len(id_list)}")
        chunk = id_list[i:i+chunk_size]

        papers = fetch_pubmed_data_given_ids(chunk)

        # store all the papers in a class
        for i, paper in enumerate (papers['PubmedArticle']):

            medline_citation = paper['MedlineCitation']

            article = medline_citation.get('Article', {})
            journal_info = article.get('Journal', {})
            journal_issue = journal_info.get('JournalIssue', {})
            pub_date = journal_issue.get('PubDate', {})

            # Extract the data from the XML
            id = medline_citation.get('PMID', 'Not Available')
            title = article.get('ArticleTitle', 'Not Available')
            abstract = article.get('Abstract', {}).get('AbstractText', 'Not Available')
            author_list = article.get('AuthorList', [])
            if isinstance(author_list, dict):
                authors = author_list.get('Author', [])
            else:
                authors = author_list

            authors = [f"{author.get('LastName', 'Not Available')} {author.get('Initials', 'Not Available')}" 
                    for author in authors if isinstance(author, dict)]

            journal = journal_info.get('Title', 'Not Available')
            publication_year = pub_date.get('Year', 'Not Available')
            publication_month = pub_date.get('Month', 'Not Available')
            publication_day = pub_date.get('Day', 'Not Available')
            doi = article.get('ELocationID', 'Not Available')
            keyword_list = medline_citation.get('KeywordList', [])
            if keyword_list and isinstance(keyword_list[0], dict):
                keywords = keyword_list[0].get('Keyword', ['Not Available'])
            else:
                keywords = ['Not Available']
            language_list = article.get('Language', ['Not Available'])

            # Create a Paper object
            paper_object = Paper(id, title, abstract, authors, journal, publication_year, publication_month, publication_day, doi, keywords, language_list)

            # Append the paper object to the list
            papers_list.append(paper_object)

            if output_path:
                if not os.path.exists(output_path):
                    os.makedirs(output_path, exist_ok=True)

                # check if the file already exists and that the id is writable
                filename = f"{output_path}/{id}.json"
                if not os.path.exists(filename) and id.isdigit():
                    paper_object.save_to_json(filename)

    return papers_list


In [6]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# journals as small letters
journals_to_include = [journal.lower() for journal in journals_to_include]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]

In [7]:
# compile the query
query = f"({keywords_inclusion[0]} OR {keywords_inclusion[1]}) AND ({keywords_inclusion[2]} OR {keywords_inclusion[3]} OR {keywords_inclusion[4]})"

if False:
    # add the exclusion criteria
    query += f" AND NOT ({keywords_exclusion[0]})"

    # add the journals to include
    query += f" AND ("
    for journal in journals_to_include:
        query += f"source:{journal} OR "
    query = query[:-4]
    query += ")"



In [8]:
query

'(nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model)'

In [9]:

# search for the query
record = article_machine(query)

# print the number of results
print(f"Number of results: {record['Count']}")

57672

 retstart now :  1000  out of :  57672

 retstart now :  2000  out of :  57672

 retstart now :  3000  out of :  57672

 retstart now :  4000  out of :  57672

 retstart now :  5000  out of :  57672

 retstart now :  6000  out of :  57672

 retstart now :  7000  out of :  57672

 retstart now :  8000  out of :  57672

 retstart now :  9000  out of :  57672

 retstart now :  10000  out of :  57672


RuntimeError: Search Backend failed: Exception:
'retstart' cannot be larger than 9998. For PubMed, ESearch can only retrieve the first 9,999 records matching the query. To obtain more than 9,999 PubMed records, consider using EDirect that contains additional logic to batch PubMed search results automatically so that an arbitrary number can be retrieved. For details see https://www.ncbi.nlm.nih.gov/books/NBK25499/

In [17]:
record

{'Count': '57672', 'RetMax': '9999', 'RetStart': '0', 'IdList': ['39224396', '39222842', '39222706', '39221994', '39221522', '39221217', '39221216', '39221205', '39220876', '39220741', '39220526', '39220405', '39220196', '39220192', '39220191', '39218565', '39217793', '39217726', '39217725', '39217616', '39217342', '39216744', '39216566', '39215953', '39214210', '39211592', '39211409', '39208936', '39215337', '39215325', '39214974', '39214218', '39213991', '39213753', '39213520', '39212620', '39212195', '39211510', '39210776', '39210734', '39210660', '39210655', '39210649', '39210348', '39209454', '39208959', '39208843', '39208350', '39208257', '39207940', '39207677', '39207557', '39207050', '39206442', '39206406', '39205875', '39205543', '39205539', '39204409', '39204402', '39204396', '39204325', '39204198', '39204071', '39204028', '39203954', '39202863', '39202513', '39201784', '39201657', '39201611', '39199350', '39199348', '39198544', '39198318', '39197799', '39197631', '39197450',

In [16]:

# get the ids
id_list = record['IdList']

# fetch the data
papers = fetch_pubmed_data_given_ids_in_chunks(id_list, output_path="papers")




Fetching papers 0 to 10000 out of 9999


KeyboardInterrupt: 

In [10]:
len(papers)

9999